# Chadstone Parking Occupancy Analysis

This notebook analyses the historical occupancy of the Chadstone Shopping Centre car parks.
The data is scraped on a regular schedule and published to `data/parking.csv` in this repository.

Each row records how many spaces are **occupied** and **vacant** in a given car park at a point in time.
The `total_occupied` and `total_vacant` columns show the site-wide totals for that timestamp.

We use [Polars](https://pola.rs) for data handling and the lightweight [`xy`](https://github.com/factorish/xy) library for charts.

> **Note** — `xy` charts do not render inline in VSCode. Every chart is saved to
> a self-contained HTML file in the [`charts/`](../charts) folder. Open those in a browser.

## 1. Setup

Import the libraries and prepare the output directory.

In [ ]:
from pathlib import Path

import polars as pl
import xy

# All generated charts land in the charts/ folder at the repo root.
CHARTS = Path("../charts")
CHARTS.mkdir(parents=True, exist_ok=True)

## 2. Load the data

Read the latest parking history directly from the GitHub-hosted CSV, then parse
`retrieved_at` into a proper datetime column.

In [ ]:
df = pl.read_csv(
    "https://github.com/jay-stein/chaddy-parking-scraper/blob/master/data/parking.csv?raw=true"
).with_columns(
    pl.col("retrieved_at").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
)

df.shape

## 3. Explore the data

Get a feel for the dataset: available car parks and the time range covered.

In [ ]:
df.select("car_park").unique().sort("car_park")

In [ ]:
df.select(
    pl.col("retrieved_at").min().alias("earliest"),
    pl.col("retrieved_at").max().alias("latest"),
)

## 4. Car Park B occupancy over time

Filter down to Car Park B, sort by time, then draw a line chart.
The chart is saved as a self-contained HTML file in
[`charts/carpark_b.html`](../charts/carpark_b.html).

In [ ]:
subset_df = df.filter(pl.col("car_park") == "B").sort("retrieved_at")
subset_df

In [ ]:
chart = xy.line_chart(
    xy.line(
        subset_df["retrieved_at"],
        subset_df["occupied"],
        color="#7c3aed",
        width=3,
    ),
    xy.x_axis(label="Time"),
    xy.y_axis(label="Occupied Spaces"),
    xy.tooltip(
        title="{x:%d %b %Y %H:%M}",
        format={"y": ",.0f"},
    ),
    title="Occupied Spaces in Car Park B Over Time",
)

CHARTS.joinpath("carpark_b.html").write_text(chart.to_html(), encoding="utf-8")
print("Saved -> charts/carpark_b.html")

## 5. Total occupancy across all car parks

Sum the occupied spaces over every car park per timestamp to see how busy the
whole centre is at each sample.

Saved to [`charts/total_occupancy.html`](../charts/total_occupancy.html).

In [ ]:
total_df = (
    df.group_by("retrieved_at")
    .agg(pl.sum("occupied").alias("total_occupied"))
    .sort("retrieved_at")
)
total_df

In [ ]:
total_chart = xy.line_chart(
    xy.line(
        total_df["retrieved_at"],
        total_df["total_occupied"],
        color="#0ea5e9",
        width=3,
    ),
    xy.x_axis(label="Time"),
    xy.y_axis(label="Total Occupied Spaces"),
    xy.tooltip(
        title="{x:%d %b %Y %H:%M}",
        format={"y": ",.0f"},
    ),
    title="Total Occupancy Across All Car Parks",
)

CHARTS.joinpath("total_occupancy.html").write_text(total_chart.to_html(), encoding="utf-8")
print("Saved -> charts/total_occupancy.html")

## 6. Regenerate the dashboard chart set

The 7 polished dashboard charts (grouped bars, weekday/weekend patterns, combos, etc.)
are defined in [`scripts/generate_charts.py`](../scripts/generate_charts.py). Run it from
here to refresh every HTML file in [`charts/`](../charts/) in one step.

In [ ]:
import subprocess, sys

repo_root = Path("..").resolve()
subprocess.run(
    ["uv", "run", "python", str(repo_root / "scripts" / "generate_charts.py")],
    cwd=str(repo_root),
    check=True,
)